# 05 · Merge, join, concat

Cheat sheet for combining DataFrames without silently corrupting them.

**What's in here**
- `pd.merge` with `how=`, `validate=`, `indicator=` and row-count checks
- The "merge multiplied my rows" pitfall on non-unique keys
- Different key names, index joins, multi-key merges, suffixes
- Key dtype mismatches: str vs int, datetime vs str, tz-aware vs tz-naive
- `pd.concat` rows/columns, `ignore_index`, `keys=`, misaligned columns
- `pd.merge_asof` for as-of (point-in-time) joins on timestamps
- Worked example: join hourly actuals to the forecast that was *actually available*
- `combine_first` / `update` to patch gaps from a second source
- Merge checklist

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

meters = pd.read_csv("../data/meters.csv", parse_dates=["signup_date"])
readings = pd.read_csv("../data/meter_readings_daily.csv", parse_dates=["date"])
print(meters.shape, readings.shape)
meters.head(3)

(300, 7) (107503, 3)


,meter_id,region,tariff,customer_type,annual_kwh_estimate,signup_date,has_solar
0,M100000,London,Fixed,sme,21622.0,2021-07-07,False
1,M100001,London,Fixed,residential,2286.0,2022-11-17,False
2,M100002,London,Fixed,residential,3665.0,2021-06-04,False


## Always know your key cardinality before merging

Before any merge ask: is the key unique on the left? on the right? on both? That decides
whether the row count can change. `nunique` vs `len` answers it in one line.

In [2]:
print("meters:   rows", len(meters),   "unique meter_id", meters["meter_id"].nunique())
print("readings: rows", len(readings), "unique meter_id", readings["meter_id"].nunique())
print("meters in readings but not in meters table:",
      set(readings["meter_id"]) - set(meters["meter_id"]))
print("meters with no readings:",
      len(set(meters["meter_id"]) - set(readings["meter_id"])))

meters:   rows 300 unique meter_id 300
readings: rows 107503 unique meter_id 301
meters in readings but not in meters table: {'M999999'}
meters with no readings: 0


## `how=` — inner / left / right / outer

`inner` keeps only matching keys, `left` keeps every left row, `outer` keeps everything.
Print the row count after each: a `left` merge on a many-to-one key must return **exactly**
`len(left)` rows. If it doesn't, the right side had duplicate keys.

In [3]:
for how in ["inner", "left", "right", "outer"]:
    m = pd.merge(readings, meters, on="meter_id", how=how)
    print(f"{how:6s} -> {len(m):7d} rows   (left had {len(readings)}, right had {len(meters)})")

inner  ->  107303 rows   (left had 107503, right had 300)
left   ->  107503 rows   (left had 107503, right had 300)
right  ->  107303 rows   (left had 107503, right had 300)
outer  ->  107503 rows   (left had 107503, right had 300)


## `validate=` — make pandas enforce the cardinality you assumed

`validate="many_to_one"` raises if the right key is not unique. This is the cheapest
insurance in pandas: write it every time you merge a fact table onto a dimension table.

**Interview check:** *"What would happen to your row count if a meter appeared twice in `meters`?"*

In [4]:
ok = pd.merge(readings, meters, on="meter_id", how="left", validate="many_to_one")
print("validated merge:", ok.shape)

# now break it: duplicate one meter in the dimension table
meters_dup = pd.concat([meters, meters.iloc[[0]]], ignore_index=True)
try:
    pd.merge(readings, meters_dup, on="meter_id", how="left", validate="many_to_one")
except Exception as e:
    print(type(e).__name__, "->", str(e)[:90])

validated merge: (107503, 9)
MergeError -> Merge keys are not unique in right dataset; not a many-to-one merge


## **Pitfall:** merging on a non-unique key silently multiplies rows

Without `validate=`, the same merge just runs and returns *more* rows than you started with.
Readings for meter M100000 now appear twice, and every aggregate downstream is inflated.

In [5]:
bad = pd.merge(readings, meters_dup, on="meter_id", how="left")
print("rows before", len(readings), "  rows after", len(bad), "  extra", len(bad) - len(readings))
print("M100000 readings before:", (readings.meter_id == "M100000").sum(),
      " after:", (bad.meter_id == "M100000").sum())
print("total kWh before: %.0f   after: %.0f" % (readings.kwh.sum(), bad.kwh.sum()))

rows before 107503   rows after 107862   extra 359
M100000 readings before: 359  after: 718
total kWh before: 1695749   after: 1717688


## `indicator=True` — find the orphans on both sides

Adds a `_merge` column with `left_only` / `right_only` / `both`. Use it with `how="outer"`
to audit what did not match, then decide deliberately what to do with those rows.

In [6]:
audit = pd.merge(readings, meters, on="meter_id", how="outer", indicator=True)
print(audit["_merge"].value_counts())
print("\nreadings with no meter record:", audit.loc[audit._merge == "left_only", "meter_id"].unique())
print("meters with no readings:      ", audit.loc[audit._merge == "right_only", "meter_id"].tolist()[:10])

_merge
both          107303
left_only        200
right_only         0
Name: count, dtype: int64

readings with no meter record: ['M999999']
meters with no readings:       []


## Different key names: `left_on` / `right_on`

Both key columns are kept in the result (with different names). Drop one afterwards, or rename
before merging so the keys match.

In [7]:
customers = meters.rename(columns={"meter_id": "mpan"})[["mpan", "region", "tariff"]]
m = pd.merge(readings, customers, left_on="meter_id", right_on="mpan", how="left")
print(m.columns.tolist())
m = m.drop(columns="mpan")
m.head(3)

['meter_id', 'date', 'kwh', 'mpan', 'region', 'tariff']


,meter_id,date,kwh,region,tariff
0,M100000,2023-01-01,96.882,London,Fixed
1,M100000,2023-01-02,111.294,London,Fixed
2,M100000,2023-01-03,64.984,London,Fixed


## Merging on the index: `join` and `left_index` / `right_index`

`df.join(other)` is a left join on the index by default. Equivalent to
`pd.merge(df, other, left_index=True, right_index=True, how="left")`.

In [8]:
meters_idx = meters.set_index("meter_id")
per_meter = readings.groupby("meter_id")["kwh"].agg(["sum", "count"])

joined = per_meter.join(meters_idx[["region", "customer_type"]], how="left")
print(joined.shape)
joined.head()

(301, 4)


,sum,count,region,customer_type
meter_id,,,,
M100000,21939.204,359,London,sme
M100001,2336.996,358,London,residential
M100002,3639.039,359,London,residential
M100003,2567.712,356,Scotland,residential
M100004,2206.994,358,Midlands,residential


## Multi-key merges

Pass a list to `on=`. Typical when the key is (entity, timestamp). Check uniqueness of the
*combination*, not the individual columns.

In [9]:
monthly = readings.assign(month=readings["date"].dt.to_period("M"))
monthly = monthly.groupby(["meter_id", "month"], as_index=False)["kwh"].sum()
print("combo unique:", not monthly.duplicated(["meter_id", "month"]).any())

# a fake second table keyed on the same combination, e.g. monthly bills
bills = monthly[["meter_id", "month"]].copy()
bills["bill_gbp"] = (monthly["kwh"] * 0.28).round(2)
pd.merge(monthly, bills, on=["meter_id", "month"], validate="one_to_one").head(3)

combo unique: True


,meter_id,month,kwh,bill_gbp
0,M100000,2023-01,2577.640,721.74
1,M100000,2023-02,2200.010,616.00
2,M100000,2023-03,2196.641,615.06


## Overlapping column names: `suffixes`

Columns present in both frames (and not used as keys) get `_x` / `_y` by default. Set explicit
suffixes so you can still tell which side a column came from a week later.

In [10]:
est = meters[["meter_id", "annual_kwh_estimate"]].rename(columns={"annual_kwh_estimate": "kwh"})
actual = readings.groupby("meter_id", as_index=False)["kwh"].sum()
cmp = pd.merge(actual, est, on="meter_id", suffixes=("_actual", "_estimate"))
cmp["ratio"] = cmp["kwh_actual"] / cmp["kwh_estimate"]
cmp.describe().round(2)

,kwh_actual,kwh_estimate,ratio
count,300.00,294.00,294.00
mean,5641.24,5638.33,1.01
std,7292.31,7307.01,0.02
min,938.29,930.00,0.97
25%,2747.89,2667.25,1.00
50%,3330.73,3290.00,1.01
75%,4047.42,3989.75,1.02
max,37316.06,36818.00,1.06


## **Pitfall:** key dtype mismatches

A merge on `"100000"` (str) vs `100000` (int), or datetime vs ISO string, matches nothing. pandas 2
at least raises `ValueError` for these; older versions returned an **empty result with no error**, and
subtler mismatches (`"100000"` vs `"100000 "`, `100000` vs `100000.0`) still match nothing silently.
Always print `dtypes` of both keys first, and check the matched row count.

In [11]:
left = pd.DataFrame({"id": ["1", "2", "3"], "a": [10, 20, 30]})
right = pd.DataFrame({"id": [1, 2, 3], "b": [1.0, 2.0, 3.0]})
print("left key:", left.id.dtype, " right key:", right.id.dtype)

try:
    m = pd.merge(left, right, on="id")
    print("str vs int -> rows:", len(m))
except Exception as e:
    print(type(e).__name__, "->", str(e)[:100])

right["id"] = right["id"].astype(str)          # fix: align dtypes explicitly
print("after astype(str) -> rows:", len(pd.merge(left, right, on="id")))

left key: object  right key: int64
ValueError -> You are trying to merge on object and int64 columns for key 'id'. If you wish to proceed you should 
after astype(str) -> rows: 3


In [12]:
# datetime vs string keys
hourly = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
as_str = pd.DataFrame({"time": ["2022-01-01 00:00:00+00:00", "2022-01-01 01:00:00+00:00"], "flag": [1, 1]})
print("hourly key:", hourly.time.dtype, "| as_str key:", as_str.time.dtype)
try:
    print(len(pd.merge(hourly, as_str, on="time")))
except Exception as e:
    print(type(e).__name__, "->", str(e)[:100])

as_str["time"] = pd.to_datetime(as_str["time"], utc=True)
print("after to_datetime(utc=True) -> rows:", len(pd.merge(hourly, as_str, on="time")))

hourly key: datetime64[ns, UTC] | as_str key: object
ValueError -> You are trying to merge on datetime64[ns, UTC] and object columns for key 'time'. If you wish to pro
after to_datetime(utc=True) -> rows: 2


## **Pitfall:** tz-aware vs tz-naive keys

pandas refuses to merge a `datetime64[ns, UTC]` key with a naive `datetime64[ns]` key. Good — but
the fix is *not* to blindly strip the tz; decide which zone the naive data was in and localise it.

In [13]:
naive = pd.DataFrame({"time": pd.date_range("2022-01-01", periods=3, freq="h"), "x": [1, 2, 3]})
print("naive:", naive.time.dtype, "| aware:", hourly.time.dtype)
try:
    pd.merge(hourly, naive, on="time")
except Exception as e:
    print(type(e).__name__, "->", str(e)[:120])

naive["time"] = naive["time"].dt.tz_localize("UTC")   # we know this source was in UTC
print("after tz_localize -> rows:", len(pd.merge(hourly, naive, on="time")))

naive: datetime64[ns] | aware: datetime64[ns, UTC]
ValueError -> You are trying to merge on datetime64[ns, UTC] and datetime64[ns] columns for key 'time'. If you wish to proceed you sho
after tz_localize -> rows: 3


## `pd.concat` — stacking rows (axis=0)

Concatenating rows keeps the original index labels, so you get duplicate index values unless you
pass `ignore_index=True`. `keys=` gives a MultiIndex that records which piece each row came from.

In [14]:
y22 = hourly[hourly.time.dt.year == 2022]
y23 = hourly[hourly.time.dt.year == 2023]

stacked = pd.concat([y22, y23])
print("index unique after plain concat:", stacked.index.is_unique, "| rows:", len(stacked))

stacked = pd.concat([y22, y23], ignore_index=True)
print("with ignore_index:", stacked.index.is_unique)

with_keys = pd.concat({"y2022": y22, "y2023": y23}, names=["source", "row"])
with_keys.groupby(level="source").size()

index unique after plain concat: True | rows: 17520
with ignore_index: True


source
y2022    8760
y2023    8760
dtype: int64

## `pd.concat` — side by side (axis=1) aligns on the index

Column-wise concat is an outer join on the index. Misaligned indexes produce NaN rows/columns,
which is a common silent source of "why do I suddenly have NaNs".

In [15]:
a = pd.Series([1, 2, 3], index=[0, 1, 2], name="a")
b = pd.Series([10, 20, 30], index=[1, 2, 3], name="b")
side = pd.concat([a, b], axis=1)
print(side)
print("\njoin='inner' keeps only the overlap:")
print(pd.concat([a, b], axis=1, join="inner"))

     a     b
0  1.0   NaN
1  2.0  10.0
2  3.0  20.0
3  NaN  30.0

join='inner' keeps only the overlap:
   a   b
1  2  10
2  3  20


In [16]:
# rows with different column sets -> NaN filled
p1 = pd.DataFrame({"time": [1, 2], "consumption": [10, 11]})
p2 = pd.DataFrame({"time": [3, 4], "consumption": [12, 13], "temp": [5.0, 6.0]})
pd.concat([p1, p2], ignore_index=True)

,time,consumption,temp
0,1,10,NaN
1,2,11,NaN
2,3,12,5.0
3,4,13,6.0


## `pd.merge_asof` — join to the *latest available* record

For each left timestamp find the most recent right record at or before it (`direction="backward"`).
This is the point-in-time join you need for forecasts, prices, fundamentals: anything published
with a lag. Both sides **must be sorted** on the key. `tolerance=` refuses stale matches,
`by=` restricts matching within a group.

**Interview check:** *"Why is `merge_asof` safer than a plain merge on the timestamp for forecast data?"*

In [17]:
fc = pd.read_csv("../data/weather_forecasts.csv", parse_dates=["origin_datetime", "forecast_datetime"])
print(fc.shape)
print(fc.dtypes)
fc.head(3)

(70004, 4)
origin_datetime      datetime64[ns, UTC]
forecast_datetime    datetime64[ns, UTC]
horizon_h                          int64
temp_forecast_c                  float64
dtype: object


,origin_datetime,forecast_datetime,horizon_h,temp_forecast_c
0,2022-01-01 00:00:00+00:00,2022-01-01 01:00:00+00:00,1,0.51
1,2022-01-01 00:00:00+00:00,2022-01-01 02:00:00+00:00,2,-1.43
2,2022-01-01 00:00:00+00:00,2022-01-01 03:00:00+00:00,3,0.40


In [18]:
# how many forecasts exist for one target hour? (issued at different origins)
one_hour = fc[fc.forecast_datetime == "2022-06-15 15:00+00:00"]
one_hour

,origin_datetime,forecast_datetime,horizon_h,temp_forecast_c
15782,2022-06-14 00:00:00+00:00,2022-06-15 15:00:00+00:00,39,19.76
15818,2022-06-14 12:00:00+00:00,2022-06-15 15:00:00+00:00,27,18.46
15854,2022-06-15 00:00:00+00:00,2022-06-15 15:00:00+00:00,15,20.64
15890,2022-06-15 12:00:00+00:00,2022-06-15 15:00:00+00:00,3,16.83


Naively merging `hourly` with `fc` on `time == forecast_datetime` multiplies rows ×4 (one per origin).
Which of those four forecasts would a trader actually have had? Depends on *when the decision is made*.

In [19]:
naive_join = pd.merge(hourly, fc, left_on="time", right_on="forecast_datetime", how="inner")
print("hourly rows:", len(hourly), " naive join rows:", len(naive_join), " ratio: %.2f" % (len(naive_join) / len(hourly)))

hourly rows: 17520  naive join rows: 70004  ratio: 4.00


### Worked example: forecasts available at a day-ahead decision time

Suppose we bid in the day-ahead auction at **12:00 UTC on day D-1** for every hour of day D.
For each target hour we need the forecast with the *latest origin ≤ decision time*. Steps:

1. Compute the decision time for each target hour (12:00 the day before).
2. Filter forecasts to `origin_datetime <= decision_time`.
3. Among remaining forecasts for that hour, keep the latest origin.

In [20]:
target = hourly[["time", "temp_c"]].copy()
target["decision_time"] = target["time"].dt.normalize() - pd.Timedelta(hours=12)   # 12:00 on D-1

cand = pd.merge(target, fc, left_on="time", right_on="forecast_datetime", how="left")
cand = cand[cand["origin_datetime"] <= cand["decision_time"]]          # only what existed at decision time
cand = cand.sort_values(["time", "origin_datetime"]).drop_duplicates("time", keep="last")

avail = target.merge(cand[["time", "origin_datetime", "horizon_h", "temp_forecast_c"]], on="time", how="left",
                     validate="one_to_one")
print("rows:", len(avail), "| hours without an admissible forecast:", avail.temp_forecast_c.isna().sum(),
      "(the first day: no forecast issued before the data starts)")
avail.iloc[24:27]

rows: 17520 | hours without an admissible forecast: 24 (the first day: no forecast issued before the data starts)


,time,temp_c,decision_time,origin_datetime,horizon_h,temp_forecast_c
24,2022-01-02 00:00:00+00:00,0.15,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,12.0,0.31
25,2022-01-02 01:00:00+00:00,-0.59,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,13.0,-1.74
26,2022-01-02 02:00:00+00:00,-1.09,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,14.0,-1.35


In [21]:
# sanity: horizons should be 12..35h for a 12:00 D-1 decision; error should grow with horizon
print(avail["horizon_h"].describe()[["min", "max"]])
avail["err"] = avail["temp_forecast_c"] - avail["temp_c"]
avail.groupby("horizon_h")["err"].agg(["mean", "std"]).round(2).iloc[::6]

min    12.0
max    35.0
Name: horizon_h, dtype: float64


,mean,std
horizon_h,,
12.0,0.28,1.10
18.0,0.29,1.44
24.0,0.25,1.84
30.0,0.46,2.20


### Same thing with `merge_asof`

Reshape forecasts so each row is (forecast_datetime, origin_datetime) and use the *origin* as the
as-of key: for each target hour, take the latest origin ≤ decision time, matching `by=` target hour.
Sorting on the `on` key is mandatory: `merge_asof` raises otherwise.

In [22]:
left = target[["time", "decision_time"]].sort_values("decision_time")
right = fc.rename(columns={"forecast_datetime": "time"}).sort_values("origin_datetime")

asof = pd.merge_asof(left, right, left_on="decision_time", right_on="origin_datetime",
                     by="time", direction="backward", tolerance=pd.Timedelta("36h"))
asof = asof.sort_values("time")
check = asof.set_index("time")["temp_forecast_c"].equals(avail.set_index("time")["temp_forecast_c"])
print("identical to the manual method:", check)
asof.iloc[24:27]

identical to the manual method: True


,time,decision_time,origin_datetime,horizon_h,temp_forecast_c
43,2022-01-02 00:00:00+00:00,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,12.0,0.31
39,2022-01-02 01:00:00+00:00,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,13.0,-1.74
40,2022-01-02 02:00:00+00:00,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,14.0,-1.35


In [23]:
# merge_asof requires sorted keys
try:
    pd.merge_asof(left.sample(frac=1, random_state=0), right, left_on="decision_time",
                  right_on="origin_datetime", by="time")
except Exception as e:
    print(type(e).__name__, "->", e)

ValueError -> left keys must be sorted


## `combine_first` / `update` — patch gaps from a second source

`a.combine_first(b)` keeps `a` where present and fills NaN from `b` (aligned on index & columns).
`a.update(b)` overwrites `a` in place with non-NaN values from `b`. Use them for
"primary feed with holes, backup feed for gaps".

In [24]:
raw = pd.read_csv("../data/hourly_power_raw.csv")
raw["time"] = pd.to_datetime(raw["time"], utc=True)
raw = raw.drop_duplicates("time").set_index("time").sort_index()
raw.loc[raw.temp_c == -999, "temp_c"] = np.nan

clean = hourly.set_index("time")
print("temp NaN in raw:", raw.temp_c.isna().sum())

patched = raw[["temp_c"]].combine_first(clean[["temp_c"]])   # fills NaN AND adds the missing hours
print("after combine_first: NaN", patched.temp_c.isna().sum(), "| rows", len(patched), "(raw had", len(raw), ")")

# update: same shape, in place, only where the other side is not NaN
raw2 = raw[["temp_c"]].copy()
raw2.update(clean[["temp_c"]])
print("after update:        NaN", raw2.temp_c.isna().sum(), "| rows", len(raw2))

temp NaN in raw: 212
after combine_first: NaN 0 | rows 17520 (raw had 17442 )
after update:        NaN 0 | rows 17442


## Merge checklist

Before you trust a merged frame:

1. **Key dtypes match** on both sides (`left[key].dtype`, `right[key].dtype`; tz-aware vs naive).
2. **Cardinality known**: `nunique()` vs `len()` on each side; pass `validate=`.
3. **Row count expectation stated** before running: left merge many-to-one → `len(result) == len(left)`.
4. **Unmatched rows inspected** with `indicator=True` and `how="outer"`, not silently dropped.
5. **Time semantics**: is the right-hand record something that was *known at* the left timestamp?
   If it has a publication/origin time, use `merge_asof` on that, not a plain merge on the target time.
6. **Suffixes explicit**, no `_x` / `_y` left behind.
7. **NaNs after merge**: are they "no match" or genuine missing values? They look the same.
8. **Sorted** before `merge_asof`, `shift`, `rolling`.